# sEEG Multi-Class CSP Analysis

This notebook implements a 4-class CSP (Common Spatial Pattern) pipeline:
1. Load all trials from 4 class folders in `dataset/`
2. Detect and remove bad channels (concatenated across all trials)
3. Apply notch + bandpass filtering and compute LMP
4. Build overlapping windows with class labels
5. Extract One-vs-Rest FBCSP features (HFB + LMP)
6. Visualize CSP spatial patterns and feature distributions

## Step 0 - Imports and Function Definitions
Define all preprocessing, CSP, and visualization functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch
from pathlib import Path


def detect_bad_channels_stat(data, standard_range=(-500, 500)):
    """Detect bad sEEG channels using statistical features.

    Args:
        data: Array with shape (n_channels, n_samples).
        standard_range: Acceptable amplitude range in microvolts.
    """
    bad_channels_indices = []

    constant_mask = np.all(np.isclose(data, data[:, [0]], atol=1e-5), axis=1)
    constant_channels = np.where(constant_mask)[0]
    bad_channels_indices.extend(constant_channels)

    means = np.mean(data, axis=1)
    offset_mask = np.abs(means) > 20
    offset_channels = np.where(offset_mask)[0]
    bad_channels_indices.extend(offset_channels)

    outside_mask = np.any(
        (data < standard_range[0]) | (data > standard_range[1]),
        axis=1,
    )
    outside_channels = np.where(outside_mask)[0]
    bad_channels_indices.extend(outside_channels)

    unique_bad_channels = sorted(set(bad_channels_indices))

    print(f"Constant channels: {constant_channels}")
    print(f"Offset channels: {offset_channels}")
    print(f"Outside standard range: {outside_channels}")

    return unique_bad_channels


def bandpass_filter(data, low_freq, high_freq, fs=2000, order=4):
    """Apply a band-pass filter to extract a target frequency band.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        low_freq: Low cutoff frequency in Hz.
        high_freq: High cutoff frequency in Hz.
        fs: Sampling rate in Hz (default: 2000).
        order: Filter order (default: 4).

    Returns:
        Filtered data with the same shape as input.
    """
    nyquist = fs / 2
    low = low_freq / nyquist
    high = high_freq / nyquist

    b, a = butter(order, [low, high], btype="band")
    filtered_data = filtfilt(b, a, data, axis=1)

    return filtered_data


def apply_notch_filter(data, freq=50, fs=2000, q=30):
    """Apply a notch filter to remove power-line interference.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        freq: Frequency to remove in Hz (50 or 60 typically).
        fs: Sampling rate in Hz.
        q: Quality factor; higher values produce a narrower notch.

    Returns:
        Filtered data with the same shape as input.
    """
    nyquist = fs / 2
    w0 = freq / nyquist

    b, a = iirnotch(w0, q)
    filtered_data = filtfilt(b, a, data, axis=1)

    return filtered_data


def create_windows(data, window_size, stride):
    """Create overlapping windows from channel-by-time data.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        window_size: Window length in samples.
        stride: Step between consecutive windows in samples.

    Returns:
        3D array with shape (n_windows, n_channels, window_size).
    """
    _, n_samples = data.shape
    windows = []
    for i in range(0, n_samples - window_size + 1, stride):
        windows.append(data[:, i : i + window_size])
    return np.array(windows)


def calculate_lmp(data, window_ms=250, fs=2000):
    """Calculate Local Motor Potential (LMP) as a running average.

    Args:
        data: 2D array with shape (n_channels, n_samples).
        window_ms: Window size for running average in milliseconds.
        fs: Sampling rate in Hz.

    Returns:
        LMP data with the same shape as input.
    """
    window_size = int(window_ms * fs / 1000)
    window = np.ones(window_size) / window_size
    lmp_data = np.apply_along_axis(
        lambda x: np.convolve(x, window, mode="same"),
        axis=1,
        arr=data,
    )
    return lmp_data


def filter_seeg_data(data, fs=2000):
    """Run the full sEEG filtering pipeline.

    Steps:
        1) Apply notch filtering to remove power-line noise.
        2) Apply band-pass filtering to obtain low/high frequency bands.
        3) Calculate Local Motor Potential (LMP) as running average.
    """
    data_no_notch = apply_notch_filter(data, freq=50, fs=fs)
    lfb_result = bandpass_filter(data_no_notch, 0.5, 30, fs)
    hfb_result = bandpass_filter(data_no_notch, 80, 200, fs)
    lmp_result = calculate_lmp(data_no_notch, window_ms=250, fs=fs)
    return lfb_result, hfb_result, lmp_result


def compute_covariance_matrices(windows, labels):
    """Compute normalized covariance matrices for each class.

    Args:
        windows: 3D array (n_windows, n_channels, n_samples).
        labels: 1D array of class labels (0 or 1).

    Returns:
        Tuple of (cov_class0, cov_class1) covariance matrices.
    """
    classes = np.unique(labels)
    cov_matrices = []
    for c in classes:
        class_windows = windows[labels == c]
        cov_sum = np.zeros((class_windows.shape[1], class_windows.shape[1]))
        for w in class_windows:
            cov_sum += np.dot(w, w.T)
        cov_matrices.append(cov_sum / np.trace(cov_sum))
    return cov_matrices[0], cov_matrices[1]


def compute_csp_filters(cov0, cov1, n_components=4):
    """Compute CSP spatial filters from two covariance matrices.

    Args:
        cov0: Covariance matrix for class 0.
        cov1: Covariance matrix for class 1.
        n_components: Number of filter pairs to return (default 4).

    Returns:
        Spatial filter matrix (n_components, n_channels).
    """
    cov_sum = cov0 + cov1
    eigvals, eigvecs = np.linalg.eigh(cov_sum)
    eigvals = eigvals[::-1]
    eigvecs = eigvecs[:, ::-1]

    whitening_matrix = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    cov0_white = whitening_matrix @ cov0 @ whitening_matrix.T

    eigvals_white, eigvecs_white = np.linalg.eigh(cov0_white)
    eigvals_white = eigvals_white[::-1]
    eigvecs_white = eigvecs_white[:, ::-1]

    csp_filters = eigvecs_white.T @ whitening_matrix

    first_last = []
    half = n_components // 2
    for i in range(half):
        first_last.append(i)
    for i in range(-half, 0):
        first_last.append(i)

    return csp_filters[first_last]


def apply_csp(windows, csp_filters):
    """Apply CSP filters and return log-variance features.

    Args:
        windows: 3D array (n_windows, n_channels, n_samples).
        csp_filters: Spatial filter matrix (n_components, n_channels).

    Returns:
        Log-variance features (n_windows, n_components).
    """
    projected = np.einsum("fc,wcs->wfs", csp_filters, windows)
    variances = np.var(projected, axis=2)
    log_var = np.log(variances)
    return log_var


def extract_ovr_fbcsp_features(hfb_windows, lmp_windows, labels, n_classes=4, n_components=4):
    """Extract One-vs-Rest Filter Bank CSP features for multi-class classification.

    For each class i, builds a binary split: class i vs all other classes.
    Computes CSP filters on HFB and LMP for each binary split,
    then concatenates all log-variance features.

    Args:
        hfb_windows: HFB windowed data (n_windows, n_channels, n_samples).
        lmp_windows: LMP windowed data (n_windows, n_channels, n_samples).
        labels: Class labels (n_windows,) with values in [0, n_classes-1].
        n_classes: Number of classes (default 4).
        n_components: Number of CSP filter pairs per band per OvR classifier.

    Returns:
        Dictionary with:
            'features': (n_windows, n_classes * 2_bands * n_components)
            'filters': dict keyed by (ovr_class, band_name)
            'band_names': list of band names
    """
    bands = {"HFB": hfb_windows, "LMP": lmp_windows}
    band_names = list(bands.keys())
    all_features = []
    all_filters = {}

    for cls in range(n_classes):
        binary_labels = (labels == cls).astype(int)

        for band_name, band_windows in bands.items():
            cov0, cov1 = compute_covariance_matrices(band_windows, binary_labels)
            filters = compute_csp_filters(cov0, cov1, n_components=n_components)
            features = apply_csp(band_windows, filters)
            all_features.append(features)
            all_filters[(cls, band_name)] = filters

    combined_features = np.hstack(all_features)
    return {
        "features": combined_features,
        "filters": all_filters,
        "band_names": band_names,
    }


def visualize_csp_filters(csp_filters, title="", step=50):
    """Visualize CSP spatial patterns as channel weight bar plots.

    Args:
        csp_filters: Filter matrix (n_components, n_channels).
        title: Title for the figure.
        step: Only show every `step`-th channel.
    """
    n_components = csp_filters.shape[0]
    n_channels = csp_filters.shape[1]

    fig, axes = plt.subplots(1, n_components, figsize=(4 * n_components, 4))
    if n_components == 1:
        axes = [axes]

    channel_indices = np.arange(0, n_channels, step)
    for i, ax in enumerate(axes):
        weights = csp_filters[i, channel_indices]
        colors_bar = ["red" if w < 0 else "blue" for w in weights]
        ax.bar(range(len(channel_indices)), weights, color=colors_bar)
        ax.set_title(f"Filter {i + 1}")
        ax.set_xlabel(f"Channel (every {step}-th)")
        ax.set_ylabel("Weight")
        ax.axhline(y=0, color="black", linewidth=0.5)

    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


def visualize_csp_features_multiclass(features, labels, band_names, class_names):
    """Visualize CSP log-variance features for all classes.

    Args:
        features: CSP features (n_windows, n_components_total).
        labels: Class labels (n_windows,).
        band_names: List of band names used.
        class_names: List of class display names.
    """
    n_components = features.shape[1]
    n_classes = len(class_names)
    n_bands = len(band_names)
    features_per_ovr = n_components // n_classes

    n_cols = min(n_components, 4)
    n_rows = int(np.ceil(n_components / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3 * n_rows))
    if n_components == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    cmap = plt.cm.tab10

    for i in range(n_components):
        ax = axes[i]
        ovr_idx = i // features_per_ovr
        feature_idx_in_ovr = i % features_per_ovr
        band_idx = feature_idx_in_ovr // (features_per_ovr // n_bands) if n_bands > 1 else 0
        band_idx = min(band_idx, n_bands - 1)

        for c in range(n_classes):
            ax.hist(features[labels == c, i], bins=15, alpha=0.5,
                    label=class_names[c], color=cmap(c))

        ax.set_title(f"Class {ovr_idx} vs Rest - {band_names[band_idx]} (F{feature_idx_in_ovr + 1})")
        ax.set_xlabel("Log-Variance")
        ax.set_ylabel("Count")
        ax.legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (12, 4)

## Step 1 - Load All Class Data
Load `.npz` files from each class folder. Each file is one trial with shape `(channels, samples)`.

In [ ]:
dataset_dir = Path('dataset')

class_configs = [
    ('class_1_关一下灯', '关一下灯'),
    ('class_2_我冷了', '我冷了'),
    ('class_3_我想出门走走', '我想出门走走'),
    ('class_4_现在几点了', '现在几点了'),
]

class_names = [cfg[1] for cfg in class_configs]
all_trials = []

for cls_idx, (folder_name, display_name) in enumerate(class_configs):
    folder = dataset_dir / folder_name
    npz_files = sorted(folder.glob('*.npz'))
    trials = []
    for f in npz_files:
        data = np.load(f)
        arr = data[data.files[0]]
        if arr.shape[0] > arr.shape[1]:
            arr = arr.T
        trials.append(arr)
    all_trials.append(trials)
    print(f'Class {cls_idx} ({display_name}): {len(trials)} trials, shape {trials[0].shape if trials else "N/A"}')

n_classes = len(all_trials)
n_channels = all_trials[0][0].shape[0]
print(f'\nTotal: {n_classes} classes, {n_channels} channels, {sum(len(t) for t in all_trials)} trials')

## Step 2 - Detect and Remove Bad Channels
Concatenate all trials along the time axis to get a robust estimate of bad channels, then remove them from every trial.

In [ ]:
all_trials_flat = [trial for cls_trials in all_trials for trial in cls_trials]
concatenated = np.concatenate(all_trials_flat, axis=1)
print(f'Concatenated data shape: {concatenated.shape}')

bad_indices = detect_bad_channels_stat(concatenated)
print(f'\nBad channels ({len(bad_indices)} total): {bad_indices}')

keep_mask = np.ones(n_channels, dtype=bool)
keep_mask[bad_indices] = False

for cls_idx in range(n_classes):
    for trial_idx in range(len(all_trials[cls_idx])):
        all_trials[cls_idx][trial_idx] = all_trials[cls_idx][trial_idx][keep_mask]

n_channels_clean = all_trials[0][0].shape[0]
print(f'\nChannels after removal: {n_channels_clean} (removed {len(bad_indices)})')

## Step 3 - Filter All Trials
Apply the full preprocessing pipeline (notch -> HFB/LFB -> LMP) to every trial.

In [ ]:
fs = 2000

all_hfb_trials = []
all_lmp_trials = []

for cls_idx in range(n_classes):
    cls_hfb = []
    cls_lmp = []
    for trial in all_trials[cls_idx]:
        _, hfb, lmp = filter_seeg_data(trial, fs=fs)
        cls_hfb.append(hfb)
        cls_lmp.append(lmp)
    all_hfb_trials.append(cls_hfb)
    all_lmp_trials.append(cls_lmp)

print(f'HFB trial shapes: {[t.shape for t in all_hfb_trials[0][:3]]}...')
print(f'LMP trial shapes: {[t.shape for t in all_lmp_trials[0][:3]]}...')

## Step 4 - Create Overlapping Windows with Labels
Cut each trial into overlapping windows and assign the trial's class label to all its windows.

In [ ]:
window_size = 250
stride = 225

hfb_windows_list = []
lmp_windows_list = []
labels_list = []

for cls_idx in range(n_classes):
    for trial_hfb, trial_lmp in zip(all_hfb_trials[cls_idx], all_lmp_trials[cls_idx]):
        hfb_w = create_windows(trial_hfb, window_size=window_size, stride=stride)
        lmp_w = create_windows(trial_lmp, window_size=window_size, stride=stride)
        n_w = len(hfb_w)
        hfb_windows_list.append(hfb_w)
        lmp_windows_list.append(lmp_w)
        labels_list.append(np.full(n_w, cls_idx, dtype=int))

hfb_windows = np.concatenate(hfb_windows_list, axis=0)
lmp_windows = np.concatenate(lmp_windows_list, axis=0)
labels = np.concatenate(labels_list, axis=0)

print(f'Total windows: {len(labels)}')
print(f'HFB windows shape: {hfb_windows.shape}')
print(f'LMP windows shape: {lmp_windows.shape}')
print(f'\nWindows per class:')
for cls_idx, name in enumerate(class_names):
    count = np.sum(labels == cls_idx)
    print(f'  Class {cls_idx} ({name}): {count} windows')

## Step 5 - Multi-Class CSP (One-vs-Rest)
For each of the 4 classes, build a binary CSP classifier (class i vs all others) on both HFB and LMP bands.
This yields 4 OvR classifiers x 2 bands x 4 components = 32 features per window.

In [ ]:
n_components = 4

fbcsp_result = extract_ovr_fbcsp_features(
    hfb_windows, lmp_windows, labels,
    n_classes=n_classes, n_components=n_components
)

print(f'Combined FBCSP features shape: {fbcsp_result["features"].shape}')
print(f'Expected: ({len(labels)} windows, {n_classes} classes x {len(fbcsp_result["band_names"])} bands x {n_components} components)')
print(f'\nFilters stored for:')
for key in fbcsp_result["filters"]:
    print(f'  {key}: shape {fbcsp_result["filters"][key].shape}')

## Step 6 - Visualize CSP Results
1. **Spatial patterns**: Channel weights for each OvR classifier and band.
2. **Feature histograms**: Log-variance distribution of all 4 classes per feature component.

In [ ]:
for cls_idx in range(n_classes):
    for band_name in fbcsp_result["band_names"]:
        filters = fbcsp_result["filters"][(cls_idx, band_name)]
        title = f"Class {cls_idx} ({class_names[cls_idx]}) vs Rest - {band_name}"
        visualize_csp_filters(filters, title=title, step=50)

In [ ]:
visualize_csp_features_multiclass(
    fbcsp_result["features"], labels,
    fbcsp_result["band_names"], class_names
)

## Next Extensions (Optional)
The extracted FBCSP features can now be fed into a classifier:
- `fbcsp_result["features"]` → feature matrix (n_windows, 32)
- `labels` → target labels (n_windows,)
- Add cross-validation, train/test split, and accuracy reporting.